<a href="https://colab.research.google.com/github/Akash9888/thesis_code/blob/master/fetch_news_dataset_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_CSV_PATH = '/content/drive/MyDrive/ThesisData/all-the-news-2-1.csv'
OUTPUT_FILE = '/content/drive/MyDrive/ThesisData/news_data_small_phase1.csv'
CHUNK_SIZE = 50000

PHARMA_CORE = [
    "Eli Lilly", "Moderna", "Johnson & Johnson", "Pfizer", "AstraZeneca",
    "Merck", "Novo Nordisk", "Gilead", "AbbVie", "Amgen", "FDA", "CDC", "HHS",
    r"\bPFE\b", r"\bLLY\b", r"\bMRNA\b", r"\bJNJ\b", r"\bABBV\b", r"\bGILD\b", r"\bMRK\b", r"\bNVO\b"
]

POLICY_CONTEXT = [
    "Drug Pricing", "Insulin", "Vaccine", "Opioid", "Patent", "Healthcare Reform",
    "Medicare", "Obamacare", "ACA", "Trump", "Biden", "Sanders", "White House"
]


re_pharma = re.compile("|".join(PHARMA_CORE), re.IGNORECASE)
re_context = re.compile("|".join(POLICY_CONTEXT), re.IGNORECASE)

def smart_filter(text):
    """Keep if (Core Pharma mentioned) OR (Policy AND Context are both mentioned)"""
    text = str(text)
    has_pharma = bool(re_pharma.search(text))
    if has_pharma:
        return True

    has_policy = bool(re_context.search(text))
    has_medical_keywords = any(kw in text.lower() for kw in ["drug", "health", "medicine", "pharma"])

    return has_policy and has_medical_keywords

def process_phase1_smart():
    print(f"Starting Smart Filtered Phase 1...")

    if not os.path.exists(DRIVE_CSV_PATH):
        print(f"Error: {DRIVE_CSV_PATH} not found.")
        return

    processed_count = 0
    if os.path.exists(OUTPUT_FILE):
        print("Existing output found. Appending new matches...")

    cols = ['date', 'title', 'article', 'publication']
    total_scanned = 0
    total_saved = 0

    try:

        reader = pd.read_csv(DRIVE_CSV_PATH, usecols=cols, chunksize=CHUNK_SIZE)

        for i, chunk in enumerate(reader):
            total_scanned += len(chunk)


            chunk = chunk.rename(columns={'title': 'headline', 'article': 'description', 'publication': 'source'})
            chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce')
            chunk = chunk.dropna(subset=['date', 'headline'])

            # Date Filter (2016-2020)
            chunk = chunk[(chunk['date'] >= '2016-01-01') & (chunk['date'] <= '2020-12-31')]

            # Apply Smart Filter to both headline and description
            mask = chunk.apply(lambda row: smart_filter(row['headline']) or smart_filter(row['description']), axis=1)
            filtered_chunk = chunk[mask].copy()


            if not filtered_chunk.empty:
                total_saved += len(filtered_chunk)
                mode = 'a' if os.path.exists(OUTPUT_FILE) else 'w'
                header = not os.path.exists(OUTPUT_FILE)
                filtered_chunk.to_csv(OUTPUT_FILE, mode=mode, index=False, header=header)

            if i % 10 == 0:
                print(f"Scanned: {total_scanned} | Matches Saved: {total_saved}")

    except Exception as e:
        print(f"Crash at row {total_scanned}: {e}")

    print(f"\nFINISH: Scanned {total_scanned} rows. Saved {total_saved} relevant pharma rows.")

if __name__ == "__main__":
    process_phase1_smart()

Mounted at /content/drive
Starting Smart Filtered Phase 1...
Scanned: 50000 | Matches Saved: 4438
Scanned: 550000 | Matches Saved: 34062
Scanned: 1050000 | Matches Saved: 59700
Scanned: 1550000 | Matches Saved: 86135
Scanned: 2050000 | Matches Saved: 140438
Scanned: 2550000 | Matches Saved: 182283

FINISH: Scanned 2688879 rows. Saved 199404 relevant pharma rows.
